# HCM0204 - independent 3DGS-MCMC ablation

This Kaggle notebook keeps MCMC independent from Improved-GS. It uses the canonical random 100K initialization (`[-3r,+3r]^3`, seed 0), paper initialization semantics, and `opacity_lr=0.05`.

The safe default runs a 700-iteration smoke test and a 9K resource gate, then stops cleanly because `CONTINUE_AFTER_RESOURCE_GATE=False`. After the gate is accepted, set that flag to `True`; training resumes in separate 15K, 22.5K, and 30K subprocesses, with an integrity manifest written after every stage. Rendering/evaluation cells are no-ops until the 30K model exists.

Before running: enable a Kaggle GPU and Internet, attach the original public HCM0204 dataset (the source COLMAP camera must be `SIMPLE_RADIAL`), and optionally attach a resume directory or ZIP. Leave `EXPECTED_COMMIT=None` only while developing; pin it after the branch is finalized.

In [ ]:
from pathlib import Path
from datetime import datetime, timezone
import csv
import hashlib
import importlib.metadata as importlib_metadata
import json
import math
import os
import platform
import re
import shutil
import subprocess
import sys
import tarfile
import warnings
import zipfile

SCENE = 'HCM0204'
BRANCH = 'mdd_mcmc'
EXPECTED_COMMIT = 'eba94bd97edffe743250c95d195c161c38092f03'  # Reviewed MCMC core.
PUBLIC_SCENE_HINT = None  # Example: '/kaggle/input/.../public_set/HCM0204'
RESUME_BUNDLE = None  # Exact directory or ZIP; never selected with broad rglob.

SEED = 0
RANDOM_INIT_POINTS = 100_000
FINAL_ITERATION = 30_000
FULL_CAP = 5_100_000
SMOKE_CAP = 400_000
RUN_SMOKE = True
RUN_RESOURCE_GATE_9000 = True
CONTINUE_AFTER_RESOURCE_GATE = False
EXPORT_FINAL_MODEL = False
EXPORT_RESUME_BUNDLE = False

JPEG_QUALITY = 96
JPEG_SUBSAMPLING = 0  # Pillow: 0 means 4:4:4.
SHARPEN_SIGMA = 0.7
SHARPEN_AMOUNT = 0.3
SAFETY_CAP_MB = 340.0
HARD_LIMIT_MB = 350.0
VARIANT_NAMES = (
    'bilinear_sharp0',
    'bilinear_sharp0p3',
    'bicubic_sharp0',
    'bicubic_sharp0p3',
)

WORK_ROOT = Path('/kaggle/working')
REPO = WORK_ROOT / 'gaussian-splatting'
CLEAN_ROOT = WORK_ROOT / 'cleaned_mcmc_hcm0204'
SMOKE_MODEL_ROOT = WORK_ROOT / 'mcmc_smoke_models'
MODEL_ROOT = WORK_ROOT / 'mcmc_models'
MODEL_SCENE = MODEL_ROOT / SCENE
STATS_PATH = MODEL_SCENE / 'mcmc_stats.jsonl'
VARIANT_ROOT = WORK_ROOT / 'mcmc_render_variants'
EVAL_ROOT = WORK_ROOT / 'mcmc_evaluation'
EXPORT_ROOT = WORK_ROOT / 'mcmc_exports'

BASELINE_Q96 = {
    'variant': 'bicubic_sharp0p3',
    'SSIM': 0.8460382789373397,
    'PSNR': 24.87937183380127,
    'LPIPS': 0.08333053061117729,
    'weighted_score': 0.8070745623360078,
    'estimated_mb_434': 312.6121747,
}

if EXPECTED_COMMIT is None:
    warnings.warn('EXPECTED_COMMIT is not pinned; source reproducibility is provisional.', stacklevel=1)
gpu_probe = subprocess.run(
    [
        'nvidia-smi', '--query-gpu=name,memory.total,driver_version',
        '--format=csv,noheader',
    ],
    text=True, capture_output=True, check=False,
)
assert gpu_probe.returncode == 0 and gpu_probe.stdout.strip(), (
    'Enable a GPU Accelerator in Kaggle. nvidia-smi: ' + gpu_probe.stderr.strip()
)
GPU_INFO = gpu_probe.stdout.strip().splitlines()
print('Python:', sys.version)
print('Platform:', platform.platform())
print('GPU:', GPU_INFO)
disk = shutil.disk_usage(WORK_ROOT)
print('Working disk free GiB:', round(disk.free / 2**30, 2))
print('Full training enabled:', CONTINUE_AFTER_RESOURCE_GATE)

In [ ]:
input_root = Path('/kaggle/input')
if PUBLIC_SCENE_HINT is not None:
    scene_candidates = [Path(PUBLIC_SCENE_HINT)]
else:
    scene_candidates = sorted(
        path
        for path in input_root.rglob(SCENE)
        if (path / 'train' / 'images').is_dir()
        and (path / 'train' / 'sparse' / '0' / 'cameras.bin').is_file()
        and (path / 'train' / 'sparse' / '0' / 'images.bin').is_file()
        and (path / 'train' / 'sparse' / '0' / 'points3D.bin').is_file()
        and (path / 'test' / 'images').is_dir()
        and (path / 'test' / 'test_poses.csv').is_file()
    )

print('Structural HCM0204 candidates (camera model is checked before undistortion):')
for candidate in scene_candidates:
    print(' -', candidate)
assert scene_candidates, 'No structurally valid HCM0204 public scene was found.'

In [ ]:
if not REPO.exists():
    subprocess.run(
        [
            'git', 'clone', '--branch', BRANCH, '--single-branch',
            'https://github.com/fulx17/gaussian-splatting.git', str(REPO),
        ],
        check=True,
    )
else:
    assert (REPO / '.git').exists(), '{} is not a Git repository'.format(REPO)
    subprocess.run(['git', 'fetch', 'origin', BRANCH], cwd=REPO, check=True)
    subprocess.run(['git', 'checkout', BRANCH], cwd=REPO, check=True)

if EXPECTED_COMMIT is None:
    subprocess.run(['git', 'merge', '--ff-only', 'origin/{}'.format(BRANCH)], cwd=REPO, check=True)
else:
    subprocess.run(['git', 'checkout', '--detach', EXPECTED_COMMIT], cwd=REPO, check=True)

SUBMODULE_PATHS = (
    'submodules/diff-gaussian-rasterization',
    'submodules/simple-knn',
    'submodules/fused-ssim',
)
subprocess.run(
    ['git', 'submodule', 'update', '--init', '--recursive', *SUBMODULE_PATHS],
    cwd=REPO, check=True,
)
HEAD_COMMIT = subprocess.check_output(['git', 'rev-parse', 'HEAD'], cwd=REPO, text=True).strip()
CURRENT_BRANCH = subprocess.check_output(
    ['git', 'branch', '--show-current'], cwd=REPO, text=True
).strip()
if EXPECTED_COMMIT is None:
    assert CURRENT_BRANCH == BRANCH, 'Expected branch {}, got {}'.format(BRANCH, CURRENT_BRANCH)
else:
    assert HEAD_COMMIT == EXPECTED_COMMIT, (
        'Commit mismatch: expected {}, got {}'.format(EXPECTED_COMMIT, HEAD_COMMIT)
    )
SUBMODULE_SHAS = {
    relative: subprocess.check_output(
        ['git', 'rev-parse', 'HEAD'], cwd=REPO / relative, text=True
    ).strip()
    for relative in SUBMODULE_PATHS
}
print('Source branch:', BRANCH)
print('Checkout:', CURRENT_BRANCH or 'detached pinned commit')
print('Commit:', HEAD_COMMIT)
print('Submodule SHAs:', json.dumps(SUBMODULE_SHAS, indent=2, sort_keys=True))

In [ ]:
build_env = os.environ.copy()
build_env['MAX_JOBS'] = '2'
if shutil.which('colmap') is None:
    subprocess.run(['apt-get', 'update', '-qq'], check=True)
    subprocess.run(['apt-get', 'install', '-y', 'colmap'], check=True)

subprocess.run(
    [sys.executable, 'scripts/apply_improved_gs_rasterizer_patch.py'],
    cwd=REPO, env=build_env, check=True,
)
subprocess.run(
    [
        sys.executable, '-m', 'pip', 'install', '--no-build-isolation',
        '--no-cache-dir', '--force-reinstall',
        str(REPO / 'submodules' / 'diff-gaussian-rasterization'),
    ],
    env=build_env, check=True,
)
subprocess.run(
    [
        sys.executable, '-m', 'pip', 'install',
        str(REPO / 'submodules' / 'simple-knn'),
        str(REPO / 'submodules' / 'fused-ssim'),
        'plyfile', 'pycolmap', 'lpips',
    ],
    env=build_env, check=True,
)
RASTERIZER_PATCH_SHA256 = hashlib.sha256(
    (REPO / 'patches' / 'improved-gs-rasterizer.patch').read_bytes()
).hexdigest()
print('Dependencies and CUDA extensions installed.')
print('Rasterizer patch SHA256:', RASTERIZER_PATCH_SHA256)

In [ ]:
subprocess.run(
    [sys.executable, 'scripts/apply_improved_gs_rasterizer_patch.py', '--check-only'],
    cwd=REPO, check=True,
)
subprocess.run(
    [sys.executable, '-m', 'unittest', 'discover', '-s', 'tests', '-p', 'test_mcmc*.py', '-v'],
    cwd=REPO, check=True,
)
subprocess.run(
    [sys.executable, 'scripts/smoke_test_mcmc_rasterizer.py'],
    cwd=REPO, check=True,
)
subprocess.run(
    [
        sys.executable, '-c',
        'from diff_gaussian_rasterization import compute_relocation; print(compute_relocation)',
    ],
    cwd=REPO, check=True,
)

def package_version(distribution):
    try:
        return importlib_metadata.version(distribution)
    except importlib_metadata.PackageNotFoundError:
        return None

colmap_probe = subprocess.run(
    ['colmap', '--version'], text=True, capture_output=True, check=False
)
COLMAP_VERSION = (colmap_probe.stdout or colmap_probe.stderr).strip().splitlines()[:3]
ENVIRONMENT_INFO = {
    'python': platform.python_version(),
    'platform': platform.platform(),
    'gpu_query': GPU_INFO,
    'colmap': COLMAP_VERSION,
    'packages': {
        name: package_version(name)
        for name in ('torch', 'torchvision', 'numpy', 'Pillow', 'pycolmap', 'plyfile', 'lpips')
    },
}
print('MCMC static and CUDA checks passed in fresh subprocesses.')
print(json.dumps(ENVIRONMENT_INFO, indent=2, sort_keys=True))

In [ ]:
os.chdir(REPO)
if str(REPO) not in sys.path:
    sys.path.insert(0, str(REPO))

from PIL import Image
from preprocess import preprocess_scene, undistort_scene
from scene.colmap_loader import read_extrinsics_binary, read_intrinsics_binary

def dataset_content_fingerprint(scene_dir):
    scene_dir = Path(scene_dir)
    files = [
        scene_dir / 'train' / 'sparse' / '0' / name
        for name in ('cameras.bin', 'images.bin', 'points3D.bin')
    ]
    files.append(scene_dir / 'test' / 'test_poses.csv')
    files.extend(path for path in (scene_dir / 'train' / 'images').iterdir() if path.is_file())
    files.extend(path for path in (scene_dir / 'test' / 'images').iterdir() if path.is_file())
    files = sorted(files, key=lambda path: path.relative_to(scene_dir).as_posix())
    assert files and all(path.is_file() for path in files)
    digest = hashlib.sha256()
    digest.update(b'hcm0204-dataset-content-v1\0')
    for path in files:
        relative = path.relative_to(scene_dir).as_posix().encode('utf-8')
        digest.update(relative + b'\0' + str(path.stat().st_size).encode('ascii') + b'\0')
        with open(path, 'rb') as handle:
            for chunk in iter(lambda: handle.read(16 * 1024 * 1024), b''):
                digest.update(chunk)
    return digest.hexdigest(), len(files)

source_candidates = []
for candidate in scene_candidates:
    try:
        source_cameras = read_intrinsics_binary(
            str(candidate / 'train' / 'sparse' / '0' / 'cameras.bin')
        )
        models = sorted({camera.model for camera in source_cameras.values()})
        print('Source camera models for {}: {}'.format(candidate, models))
        if len(source_cameras) == 1 and models == ['SIMPLE_RADIAL']:
            source_candidates.append(candidate)
    except Exception as error:
        print('Rejected unreadable candidate {}: {}'.format(candidate, error))
assert len(source_candidates) == 1, (
    'Expected exactly one original SIMPLE_RADIAL HCM0204 source, found {}. '
    'Set PUBLIC_SCENE_HINT to the original public scene.'.format(len(source_candidates))
)
SCENE_SOURCE = source_candidates[0]
PUBLIC_ROOT = SCENE_SOURCE.parent
source_images = read_extrinsics_binary(
    str(SCENE_SOURCE / 'train' / 'sparse' / '0' / 'images.bin')
)
assert len(source_images) == 240, 'Original images.bin must register 240 images.'
SOURCE_DATASET_FINGERPRINT, SOURCE_FINGERPRINT_FILE_COUNT = dataset_content_fingerprint(SCENE_SOURCE)

def validate_clean_scene(scene_dir):
    scene_dir = Path(scene_dir)
    train_dir = scene_dir / 'train' / 'images'
    test_dir = scene_dir / 'test' / 'images'
    sparse_dir = scene_dir / 'train' / 'sparse' / '0'
    train_images = sorted(path for path in train_dir.iterdir() if path.is_file())
    test_images = sorted(path for path in test_dir.iterdir() if path.is_file())
    with open(scene_dir / 'test' / 'test_poses.csv', newline='', encoding='utf-8') as handle:
        test_rows = list(csv.DictReader(handle))
    cameras = read_intrinsics_binary(str(sparse_dir / 'cameras.bin'))
    registered = read_extrinsics_binary(str(sparse_dir / 'images.bin'))
    disk_names = {path.name for path in train_images}
    registered_names = {image.name for image in registered.values()}
    assert len(train_images) == 240, 'Expected 240 train images, got {}'.format(len(train_images))
    assert len(test_images) == 60, 'Expected 60 test GT images, got {}'.format(len(test_images))
    assert len(test_rows) == 60, 'Expected 60 test poses, got {}'.format(len(test_rows))
    assert len(cameras) == 1 and {camera.model for camera in cameras.values()} == {'PINHOLE'}
    assert len(registered) == 240, 'Clean images.bin must register exactly 240 images.'
    assert registered_names == disk_names, 'Clean images.bin names must exactly match train/images.'
    non_rgba = []
    for image_path in train_images:
        with Image.open(image_path) as image:
            if image.mode != 'RGBA':
                non_rgba.append(image_path.name)
    assert not non_rgba, 'All cleaned train images must be RGBA; examples: {}'.format(non_rgba[:3])
    clean_digest, fingerprint_file_count = dataset_content_fingerprint(scene_dir)
    return {
        'train_images': len(train_images),
        'registered_image_names': len(registered_names),
        'test_images': len(test_images),
        'test_poses': len(test_rows),
        'camera_models': ['PINHOLE'],
        'train_image_mode': 'RGBA',
        'content_sha256': clean_digest,
        'fingerprint_file_count': fingerprint_file_count,
    }

CLEAN_SCENE = CLEAN_ROOT / SCENE
PREPROCESS_MARKER = CLEAN_SCENE / '.preprocess_complete.json'
if CLEAN_SCENE.exists():
    assert PREPROCESS_MARKER.is_file(), (
        '{} exists without a completion marker; remove the partial output.'.format(CLEAN_SCENE)
    )
    marker = json.loads(PREPROCESS_MARKER.read_text(encoding='utf-8'))
    assert marker.get('source_dataset_fingerprint') == SOURCE_DATASET_FINGERPRINT
    DATASET_STATS = validate_clean_scene(CLEAN_SCENE)
    assert marker.get('clean_dataset_fingerprint') == DATASET_STATS['content_sha256']
else:
    CLEAN_ROOT.mkdir(parents=True, exist_ok=True)
    temporary_scene = CLEAN_ROOT / '{}.preprocessing'.format(SCENE)
    if temporary_scene.exists():
        shutil.rmtree(temporary_scene)
    shutil.copytree(SCENE_SOURCE, temporary_scene)
    undistort_scene(temporary_scene)
    preprocess_scene(temporary_scene)
    DATASET_STATS = validate_clean_scene(temporary_scene)
    marker_payload = {
        **DATASET_STATS,
        'schema_version': 2,
        'scene': SCENE,
        'source': str(SCENE_SOURCE),
        'source_dataset_fingerprint': SOURCE_DATASET_FINGERPRINT,
        'clean_dataset_fingerprint': DATASET_STATS['content_sha256'],
        'completed_utc': datetime.now(timezone.utc).isoformat(),
    }
    (temporary_scene / '.preprocess_complete.json').write_text(
        json.dumps(marker_payload, indent=2, ensure_ascii=False), encoding='utf-8'
    )
    temporary_scene.replace(CLEAN_SCENE)

DATASET_FINGERPRINT = hashlib.sha256(
    json.dumps(
        {
            'source': SOURCE_DATASET_FINGERPRINT,
            'clean': DATASET_STATS['content_sha256'],
        },
        sort_keys=True, separators=(',', ':'),
    ).encode('utf-8')
).hexdigest()
print('Original source:', SCENE_SOURCE)
print('Preprocessed scene:', CLEAN_SCENE)
print('Dataset fingerprint:', DATASET_FINGERPRINT)
print(json.dumps(DATASET_STATS, indent=2, sort_keys=True))

In [ ]:
MCMC_CONFIG = {
    'density_control': 'mcmc',
    'iterations': FINAL_ITERATION,
    'resolution': 1,
    'lambda_dssim': 0.2,
    'optimizer_type': 'default',
    'cap_max': FULL_CAP,
    'densify_from_iter': 500,
    'densify_until_iter': 25_000,
    'densification_interval': 100,
    'mcmc_init_type': 'random',
    'mcmc_random_points': RANDOM_INIT_POINTS,
    'mcmc_init_seed': SEED,
    'mcmc_random_bounds': 'world_origin_cube_[-3r,+3r]^3',
    'mcmc_init_mode': 'paper',
    'opacity_lr': 0.05,
    'mcmc_noise_lr': 500_000.0,
    'mcmc_opacity_reg': 0.01,
    'mcmc_scale_reg': 0.01,
    'mcmc_growth_rate': 1.05,
    'mcmc_min_opacity': 0.005,
    'mcmc_noise_chunk_size': 250_000,
    'resource_gate_iteration': 9_000,
    'full_stage_targets': [15_000, 22_500, 30_000],
    'seed': SEED,
}

def fingerprint(payload):
    canonical = json.dumps(payload, sort_keys=True, separators=(',', ':')).encode('utf-8')
    return hashlib.sha256(canonical).hexdigest()

def atomic_write_json(path, payload):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    temporary = Path(str(path) + '.tmp')
    temporary.write_text(
        json.dumps(payload, indent=2, ensure_ascii=False, sort_keys=True), encoding='utf-8'
    )
    os.replace(temporary, path)

CONFIG_FINGERPRINT = fingerprint(MCMC_CONFIG)
ENVIRONMENT_FINGERPRINT = fingerprint(ENVIRONMENT_INFO)
MODEL_SCENE.mkdir(parents=True, exist_ok=True)
EXPERIMENT_MANIFEST_PATH = MODEL_SCENE / 'experiment_manifest.json'
RESUME_MANIFEST_PATH = MODEL_SCENE / 'resume_manifest.json'
EXPERIMENT_MANIFEST = {
    'schema_version': 2,
    'method': 'mcmc',
    'scene': SCENE,
    'branch': BRANCH,
    'commit': HEAD_COMMIT,
    'submodule_shas': SUBMODULE_SHAS,
    'rasterizer_patch_sha256': RASTERIZER_PATCH_SHA256,
    'config': MCMC_CONFIG,
    'config_fingerprint': CONFIG_FINGERPRINT,
    'dataset': {
        **DATASET_STATS,
        'source_content_sha256': SOURCE_DATASET_FINGERPRINT,
        'combined_fingerprint': DATASET_FINGERPRINT,
    },
    'dataset_fingerprint': DATASET_FINGERPRINT,
    'environment': ENVIRONMENT_INFO,
    'environment_fingerprint': ENVIRONMENT_FINGERPRINT,
    'created_utc': datetime.now(timezone.utc).isoformat(),
}
if EXPERIMENT_MANIFEST_PATH.exists():
    previous = json.loads(EXPERIMENT_MANIFEST_PATH.read_text(encoding='utf-8'))
    assert previous['config_fingerprint'] == CONFIG_FINGERPRINT
    assert previous['dataset_fingerprint'] == DATASET_FINGERPRINT
    assert previous['environment_fingerprint'] == ENVIRONMENT_FINGERPRINT
    assert previous['commit'] == HEAD_COMMIT
    assert previous['submodule_shas'] == SUBMODULE_SHAS
    EXPERIMENT_MANIFEST = previous
else:
    atomic_write_json(EXPERIMENT_MANIFEST_PATH, EXPERIMENT_MANIFEST)
print('Config fingerprint:', CONFIG_FINGERPRINT)
print('Environment fingerprint:', ENVIRONMENT_FINGERPRINT)

In [ ]:
CHECKPOINT_PATTERN = re.compile(r'^chkpnt(\d+)\.pth$')
_FILE_SHA256_CACHE = {}

def checkpoint_iteration(path):
    match = CHECKPOINT_PATTERN.match(Path(path).name)
    return int(match.group(1)) if match else None

def sha256_file(path):
    path = Path(path)
    stat = path.stat()
    key = (str(path.resolve()), stat.st_size, stat.st_mtime_ns)
    if key not in _FILE_SHA256_CACHE:
        digest = hashlib.sha256()
        with open(path, 'rb') as handle:
            for chunk in iter(lambda: handle.read(16 * 1024 * 1024), b''):
                digest.update(chunk)
        _FILE_SHA256_CACHE[key] = digest.hexdigest()
    return _FILE_SHA256_CACHE[key]

def latest_checkpoint(model_scene):
    candidates = []
    for path in Path(model_scene).glob('chkpnt*.pth'):
        iteration = checkpoint_iteration(path)
        if iteration is not None:
            candidates.append((iteration, path))
    return max(candidates, default=(None, None), key=lambda item: item[0])

def validate_resume_manifest(manifest):
    assert manifest['schema_version'] == 2
    assert manifest['method'] == 'mcmc'
    assert manifest['scene'] == SCENE
    assert manifest['branch'] == BRANCH
    assert manifest['commit'] == HEAD_COMMIT
    assert manifest['submodule_shas'] == SUBMODULE_SHAS
    assert manifest['config_fingerprint'] == CONFIG_FINGERPRINT
    assert manifest['dataset_fingerprint'] == DATASET_FINGERPRINT
    assert manifest['environment_fingerprint'] == ENVIRONMENT_FINGERPRINT
    assert checkpoint_iteration(manifest['checkpoint_file']) == manifest['checkpoint_iteration']
    assert manifest['checkpoint_size_bytes'] > 0
    assert re.fullmatch(r'[0-9a-f]{64}', manifest['checkpoint_sha256'])

def read_stats_records(path):
    records = []
    path = Path(path)
    if not path.is_file():
        return records
    for line_number, line in enumerate(path.read_text(encoding='utf-8').splitlines(), 1):
        if not line.strip():
            continue
        try:
            records.append(json.loads(line))
        except json.JSONDecodeError as error:
            raise AssertionError('Invalid stats JSONL line {}: {}'.format(line_number, error))
    return records

def latest_run_end(path):
    endings = [record for record in read_stats_records(path) if record.get('event') == 'run_end']
    return endings[-1] if endings else None

def write_resume_manifest(checkpoint_path, stats_path):
    checkpoint_path = Path(checkpoint_path)
    iteration = checkpoint_iteration(checkpoint_path)
    assert iteration is not None
    telemetry = latest_run_end(stats_path)
    assert telemetry is not None and telemetry['iteration'] == iteration
    stats_path = Path(stats_path)
    payload = {
        'schema_version': 2,
        'method': 'mcmc',
        'scene': SCENE,
        'branch': BRANCH,
        'commit': HEAD_COMMIT,
        'submodule_shas': SUBMODULE_SHAS,
        'config_fingerprint': CONFIG_FINGERPRINT,
        'dataset_fingerprint': DATASET_FINGERPRINT,
        'environment_fingerprint': ENVIRONMENT_FINGERPRINT,
        'checkpoint_file': checkpoint_path.name,
        'checkpoint_iteration': iteration,
        'checkpoint_size_bytes': checkpoint_path.stat().st_size,
        'checkpoint_sha256': sha256_file(checkpoint_path),
        'stats_file': stats_path.name,
        'stats_size_bytes': stats_path.stat().st_size,
        'stats_sha256': sha256_file(stats_path),
        'telemetry': telemetry,
        'updated_utc': datetime.now(timezone.utc).isoformat(),
    }
    atomic_write_json(RESUME_MANIFEST_PATH, payload)
    return payload

def validate_local_resume(checkpoint_path):
    checkpoint_path = Path(checkpoint_path)
    assert RESUME_MANIFEST_PATH.is_file(), (
        'Checkpoint {} has no resume_manifest.json; refusing implicit resume.'.format(checkpoint_path)
    )
    manifest = json.loads(RESUME_MANIFEST_PATH.read_text(encoding='utf-8'))
    validate_resume_manifest(manifest)
    assert manifest['checkpoint_file'] == checkpoint_path.name
    assert manifest['checkpoint_iteration'] == checkpoint_iteration(checkpoint_path)
    assert manifest['checkpoint_size_bytes'] == checkpoint_path.stat().st_size
    assert manifest['checkpoint_sha256'] == sha256_file(checkpoint_path)
    if manifest.get('stats_file') and STATS_PATH.is_file():
        current_stats_size = STATS_PATH.stat().st_size
        assert current_stats_size >= manifest['stats_size_bytes'], (
            'Local stats are truncated relative to the last completed stage.'
        )
        if current_stats_size == manifest['stats_size_bytes']:
            assert manifest['stats_sha256'] == sha256_file(STATS_PATH)
        else:
            print('Stats contain telemetry after the last completed manifest; checkpoint remains valid.')
    return manifest

def copy_stream_atomic(source_stream, destination, expected_size, expected_sha256):
    destination = Path(destination)
    destination.parent.mkdir(parents=True, exist_ok=True)
    if destination.is_file():
        if destination.stat().st_size == expected_size and sha256_file(destination) == expected_sha256:
            return
    temporary = Path(str(destination) + '.tmp')
    temporary.unlink(missing_ok=True)
    digest = hashlib.sha256()
    total = 0
    try:
        with open(temporary, 'wb') as output:
            while True:
                chunk = source_stream.read(16 * 1024 * 1024)
                if not chunk:
                    break
                output.write(chunk)
                digest.update(chunk)
                total += len(chunk)
            output.flush()
            os.fsync(output.fileno())
        assert total == expected_size, 'Imported size mismatch: {} != {}'.format(total, expected_size)
        assert digest.hexdigest() == expected_sha256, 'Imported checkpoint SHA256 mismatch.'
        os.replace(temporary, destination)
    except Exception:
        temporary.unlink(missing_ok=True)
        raise

def import_resume_bundle(bundle):
    bundle = Path(bundle)
    assert bundle.exists(), 'Resume bundle does not exist: {}'.format(bundle)
    if bundle.is_dir():
        manifest_path = bundle / 'resume_manifest.json'
        assert manifest_path.is_file(), 'Missing {}'.format(manifest_path)
        manifest = json.loads(manifest_path.read_text(encoding='utf-8'))
        validate_resume_manifest(manifest)
        source_checkpoint = bundle / manifest['checkpoint_file']
        assert source_checkpoint.is_file(), 'Missing {}'.format(source_checkpoint)
        checkpoint_source = ('file', source_checkpoint)
        stats_source = ('file', bundle / manifest['stats_file']) if manifest.get('stats_file') else None
    else:
        assert zipfile.is_zipfile(bundle), 'RESUME_BUNDLE must be a directory or ZIP.'
        with zipfile.ZipFile(bundle) as archive:
            manifest = json.loads(archive.read('resume_manifest.json').decode('utf-8'))
        validate_resume_manifest(manifest)
        checkpoint_source = ('zip', manifest['checkpoint_file'])
        stats_source = ('zip', manifest['stats_file']) if manifest.get('stats_file') else None

    existing_iteration, existing_checkpoint = latest_checkpoint(MODEL_SCENE)
    source_iteration = manifest['checkpoint_iteration']
    if existing_checkpoint is not None and existing_iteration > source_iteration:
        validate_local_resume(existing_checkpoint)
        print('Keeping newer local checkpoint:', existing_checkpoint)
        return
    destination = MODEL_SCENE / manifest['checkpoint_file']
    if checkpoint_source[0] == 'file':
        with open(checkpoint_source[1], 'rb') as source:
            copy_stream_atomic(
                source, destination, manifest['checkpoint_size_bytes'], manifest['checkpoint_sha256']
            )
    else:
        with zipfile.ZipFile(bundle) as archive:
            info = archive.getinfo(checkpoint_source[1])
            assert info.file_size == manifest['checkpoint_size_bytes']
            with archive.open(info) as source:
                copy_stream_atomic(
                    source, destination, manifest['checkpoint_size_bytes'], manifest['checkpoint_sha256']
                )

    if stats_source is not None:
        if stats_source[0] == 'file':
            assert stats_source[1].is_file(), 'Resume stats file is missing.'
            with open(stats_source[1], 'rb') as source:
                copy_stream_atomic(source, STATS_PATH, manifest['stats_size_bytes'], manifest['stats_sha256'])
        else:
            with zipfile.ZipFile(bundle) as archive:
                info = archive.getinfo(stats_source[1])
                assert info.file_size == manifest['stats_size_bytes']
                with archive.open(info) as source:
                    copy_stream_atomic(source, STATS_PATH, manifest['stats_size_bytes'], manifest['stats_sha256'])
    atomic_write_json(RESUME_MANIFEST_PATH, manifest)
    validate_local_resume(destination)
    print('Atomically imported resume checkpoint:', destination)

def read_ply_vertex_count(path):
    vertex_count = None
    with open(path, 'rb') as handle:
        while True:
            line = handle.readline()
            if not line:
                raise RuntimeError('PLY header ended before end_header.')
            text = line.decode('ascii').strip()
            if text.startswith('element vertex '):
                vertex_count = int(text.split()[-1])
            if text == 'end_header':
                break
    assert vertex_count is not None
    return vertex_count

def stage_stats_records(stats_path, target_iteration):
    records = read_stats_records(stats_path)
    starts = [
        index for index, record in enumerate(records)
        if record.get('event') == 'run_start'
        and record.get('run_until_iteration') == target_iteration
    ]
    assert starts, 'No run_start for target {} in {}'.format(target_iteration, stats_path)
    stage = records[starts[-1]:]
    endings = [
        index for index, record in enumerate(stage)
        if record.get('event') == 'run_end' and record.get('iteration') == target_iteration
    ]
    assert endings, 'No completed run_end for target {}.'.format(target_iteration)
    return stage[:endings[0] + 1]

def assert_mcmc_stage_stats(
    stats_path, target_iteration, cap_max, expected_initial_count=None,
    require_growth=False, require_cap=False,
):
    stage = stage_stats_records(stats_path, target_iteration)
    start = stage[0]
    end = stage[-1]
    numeric_fields = (
        'gaussians', 'loss', 'l1', 'ssim', 'depth_loss', 'opacity_penalty',
        'scale_penalty', 'xyz_lr', 'before', 'after', 'added', 'relocated',
        'peak_vram_allocated_bytes', 'peak_vram_reserved_bytes',
    )
    for record in stage:
        for field in numeric_fields:
            if field in record:
                assert math.isfinite(float(record[field])), (
                    'Non-finite {} at iteration {}'.format(field, record.get('iteration'))
                )
        if 'gaussians' in record:
            assert 0 < int(record['gaussians']) <= cap_max
    density = [record for record in stage if record.get('event') == 'density_control']
    assert density, 'MCMC stage contains no density-control telemetry.'
    assert [record['iteration'] for record in density] == sorted(record['iteration'] for record in density)
    for record in density:
        assert record['cap_max'] == cap_max
        assert record['after'] == record['gaussians']
        assert record['after'] == record['before'] + record['added']
        assert 0 <= record['relocated'] <= record['before']
        assert 0 < record['after'] <= cap_max
    if expected_initial_count is not None:
        assert start['first_iteration'] == 0
        assert density[0]['before'] == expected_initial_count
    if require_growth:
        assert any(record['added'] > 0 for record in density)
        assert end['gaussians'] > (expected_initial_count or 0)
    if require_cap:
        assert end['gaussians'] == cap_max
        assert any(record['after'] == cap_max for record in density)
    assert end['iteration'] == target_iteration
    assert 0 < end['gaussians'] <= cap_max
    return {
        'target_iteration': target_iteration,
        'first_iteration': start['first_iteration'],
        'gaussians': end['gaussians'],
        'density_events': len(density),
        'first_density': density[0],
        'last_density': density[-1],
        'peak_vram_allocated_bytes': end['peak_vram_allocated_bytes'],
        'peak_vram_reserved_bytes': end['peak_vram_reserved_bytes'],
    }

def validate_manifest_telemetry(manifest, minimum_iteration, cap_max, require_cap=False):
    telemetry = manifest.get('telemetry')
    assert telemetry and telemetry['event'] == 'run_end'
    assert telemetry['iteration'] >= minimum_iteration
    assert math.isfinite(float(telemetry['gaussians']))
    assert 0 < telemetry['gaussians'] <= cap_max
    if require_cap:
        assert telemetry['gaussians'] == cap_max
    return telemetry

if RESUME_BUNDLE is not None:
    import_resume_bundle(RESUME_BUNDLE)

def build_mcmc_command(model_root, cap_max, stop_after, stats_path, start_checkpoint=None):
    command = [
        sys.executable, 'train_scene.py',
        '--input_dir', str(CLEAN_ROOT),
        '--model_dir', str(model_root),
        '--scene_name', SCENE,
        '--iterations', str(FINAL_ITERATION),
        '--resolution', '1',
        '--lambda_dssim', '0.2',
        '--opacity_lr', '0.05',
        '--test_iterations', '-1',
        '--save_iterations', str(FINAL_ITERATION),
        '--checkpoint_iterations', str(stop_after),
        '--density_control', 'mcmc',
        '--cap_max', str(cap_max),
        '--densify_from_iter', '500',
        '--densify_until_iter', '25000',
        '--densification_interval', '100',
        '--mcmc_init_type', 'random',
        '--mcmc_random_points', str(RANDOM_INIT_POINTS),
        '--mcmc_init_mode', 'paper',
        '--mcmc_noise_lr', '500000',
        '--mcmc_opacity_reg', '0.01',
        '--mcmc_scale_reg', '0.01',
        '--mcmc_growth_rate', '1.05',
        '--mcmc_min_opacity', '0.005',
        '--mcmc_noise_chunk_size', '250000',
        '--optimizer_type', 'default',
        '--stop_after_iteration', str(stop_after),
        '--checkpoint_keep_last', '1',
        '--stats_path', str(stats_path),
        '--seed', str(SEED),
    ]
    if start_checkpoint is not None:
        command.extend(['--start_checkpoint', str(start_checkpoint)])
    return command

def run_mcmc_stage(model_root, cap_max, stop_after, stats_path, allow_resume):
    model_scene = Path(model_root) / SCENE
    model_scene.mkdir(parents=True, exist_ok=True)
    current_iteration, current_checkpoint = latest_checkpoint(model_scene)
    current_manifest = None
    if current_checkpoint is not None and model_scene == MODEL_SCENE:
        current_manifest = validate_local_resume(current_checkpoint)
    if current_checkpoint is not None and not allow_resume:
        raise AssertionError('Fresh stage unexpectedly contains {}'.format(current_checkpoint))
    if current_iteration is not None and current_iteration >= stop_after:
        print('Stage {} covered by {}; skipping.'.format(stop_after, current_checkpoint))
        return {
            'checkpoint': current_checkpoint,
            'iteration': current_iteration,
            'manifest': current_manifest,
            'ran': False,
        }
    start_checkpoint = current_checkpoint if allow_resume else None
    command = build_mcmc_command(model_root, cap_max, stop_after, stats_path, start_checkpoint)
    environment = os.environ.copy()
    environment['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'
    print('Running:', ' '.join(command))
    subprocess.run(command, cwd=REPO, env=environment, check=True)
    final_iteration, final_checkpoint = latest_checkpoint(model_scene)
    assert final_checkpoint is not None and final_iteration == stop_after
    manifest = None
    if model_scene == MODEL_SCENE:
        manifest = write_resume_manifest(final_checkpoint, stats_path)
    return {
        'checkpoint': final_checkpoint,
        'iteration': final_iteration,
        'manifest': manifest,
        'ran': True,
    }

In [ ]:
SMOKE_REPORT = None
if RUN_SMOKE:
    smoke_scene = SMOKE_MODEL_ROOT / SCENE
    if smoke_scene.exists():
        shutil.rmtree(smoke_scene)
    smoke_stats = smoke_scene / 'mcmc_stats.jsonl'
    smoke_outcome = run_mcmc_stage(
        model_root=SMOKE_MODEL_ROOT,
        cap_max=SMOKE_CAP,
        stop_after=700,
        stats_path=smoke_stats,
        allow_resume=False,
    )
    assert smoke_outcome['iteration'] == 700
    assert read_ply_vertex_count(smoke_scene / 'input.ply') == RANDOM_INIT_POINTS
    SMOKE_REPORT = assert_mcmc_stage_stats(
        smoke_stats, target_iteration=700, cap_max=SMOKE_CAP,
        expected_initial_count=RANDOM_INIT_POINTS, require_growth=True, require_cap=False,
    )
    assert RANDOM_INIT_POINTS < SMOKE_REPORT['gaussians'] < SMOKE_CAP
    print('Smoke stage passed:', smoke_outcome['checkpoint'])
    print(json.dumps(SMOKE_REPORT, indent=2, sort_keys=True))
else:
    print('Smoke stage disabled by RUN_SMOKE=False.')

In [ ]:
RESOURCE_GATE_PASSED = False
GATE_REPORT = None
GATE_CHECKPOINT = None
if RUN_RESOURCE_GATE_9000:
    gate_outcome = run_mcmc_stage(
        model_root=MODEL_ROOT,
        cap_max=FULL_CAP,
        stop_after=9_000,
        stats_path=STATS_PATH,
        allow_resume=True,
    )
    GATE_CHECKPOINT = gate_outcome['checkpoint']
    if gate_outcome['ran']:
        gate_stage = stage_stats_records(STATS_PATH, 9_000)
        fresh_gate = gate_stage[0]['first_iteration'] == 0
        GATE_REPORT = assert_mcmc_stage_stats(
            STATS_PATH, target_iteration=9_000, cap_max=FULL_CAP,
            expected_initial_count=(RANDOM_INIT_POINTS if fresh_gate else None),
            require_growth=fresh_gate, require_cap=True,
        )
        if fresh_gate:
            assert read_ply_vertex_count(MODEL_SCENE / 'input.ply') == RANDOM_INIT_POINTS
    else:
        telemetry = validate_manifest_telemetry(
            gate_outcome['manifest'], minimum_iteration=9_000,
            cap_max=FULL_CAP, require_cap=True,
        )
        GATE_REPORT = {'resumed_telemetry': telemetry}
    RESOURCE_GATE_PASSED = True
    print('9K resource gate passed:', GATE_CHECKPOINT)
    print(json.dumps(GATE_REPORT, indent=2, sort_keys=True))
    disk = shutil.disk_usage(WORK_ROOT)
    print('Working disk free after 9K GiB:', round(disk.free / 2**30, 2))
    subprocess.run(['nvidia-smi'], check=True)
elif (latest_checkpoint(MODEL_SCENE)[1] is not None):
    gate_iteration, GATE_CHECKPOINT = latest_checkpoint(MODEL_SCENE)
    gate_manifest = validate_local_resume(GATE_CHECKPOINT)
    validate_manifest_telemetry(
        gate_manifest, minimum_iteration=9_000, cap_max=FULL_CAP, require_cap=True
    )
    assert gate_iteration >= 9_000
    RESOURCE_GATE_PASSED = True
    print('9K gate accepted from validated local resume:', GATE_CHECKPOINT)
else:
    print('9K resource gate disabled and no validated >=9K checkpoint is present.')

if RESOURCE_GATE_PASSED and not CONTINUE_AFTER_RESOURCE_GATE:
    print('Gate-only run complete. Later cells will skip safely. Set CONTINUE_AFTER_RESOURCE_GATE=True after review.')

In [ ]:
FINAL_PLY = MODEL_SCENE / 'point_cloud' / 'iteration_{}'.format(FINAL_ITERATION) / 'point_cloud.ply'
FULL_STAGE_REPORTS = []
FULL_MODEL_READY = FINAL_PLY.is_file()
if CONTINUE_AFTER_RESOURCE_GATE:
    assert RESOURCE_GATE_PASSED, (
        'Run or import a validated 9K resource gate before enabling full training.'
    )
    if FINAL_PLY.is_file():
        print('Final model already exists; split training stages are skipped:', FINAL_PLY)
    else:
        for target in (15_000, 22_500, 30_000):
            stage_outcome = run_mcmc_stage(
                model_root=MODEL_ROOT,
                cap_max=FULL_CAP,
                stop_after=target,
                stats_path=STATS_PATH,
                allow_resume=True,
            )
            if stage_outcome['ran']:
                report = assert_mcmc_stage_stats(
                    STATS_PATH, target_iteration=target, cap_max=FULL_CAP,
                    require_growth=False, require_cap=True,
                )
            else:
                report = {
                    'resumed_telemetry': validate_manifest_telemetry(
                        stage_outcome['manifest'], minimum_iteration=target,
                        cap_max=FULL_CAP, require_cap=True,
                    )
                }
            latest_iteration, latest_path = latest_checkpoint(MODEL_SCENE)
            assert latest_iteration >= target and latest_path == stage_outcome['checkpoint']
            assert stage_outcome['manifest'] is not None
            FULL_STAGE_REPORTS.append({'target': target, 'report': report})
            print('Completed isolated stage {}: {}'.format(target, stage_outcome['checkpoint']))
    FULL_MODEL_READY = FINAL_PLY.is_file()
    assert FULL_MODEL_READY, '30K stage completed without final point_cloud.ply.'
    assert (MODEL_SCENE / 'cfg_args').is_file()
    assert (MODEL_SCENE / 'mcmc_config.json').is_file()
    print('Full MCMC training complete:', FINAL_PLY)
else:
    print('Full training intentionally skipped: CONTINUE_AFTER_RESOURCE_GATE=False.')

In [ ]:
MODEL_AUDIT = None
FINAL_GAUSSIANS = None
if FULL_MODEL_READY:
    FINAL_GAUSSIANS = read_ply_vertex_count(FINAL_PLY)
    assert FINAL_GAUSSIANS == FULL_CAP
    assert read_ply_vertex_count(MODEL_SCENE / 'input.ply') == RANDOM_INIT_POINTS
    internal_config_path = MODEL_SCENE / 'mcmc_config.json'
    internal_config = json.loads(internal_config_path.read_text(encoding='utf-8'))
    assert internal_config['density_control'] == 'mcmc'
    assert internal_config['cap_max'] == FULL_CAP
    assert internal_config['seed'] == SEED
    assert internal_config['dataset']['mcmc_init_type'] == 'random'
    assert internal_config['dataset']['mcmc_random_points'] == RANDOM_INIT_POINTS
    assert math.isclose(internal_config['optimization']['opacity_lr'], 0.05)
    assert internal_config['optimization']['mcmc_init_mode'] == 'paper'
    MODEL_AUDIT = {
        'iteration': FINAL_ITERATION,
        'gaussians': FINAL_GAUSSIANS,
        'cap_max': FULL_CAP,
        'random_init_points': RANDOM_INIT_POINTS,
        'ply_size_bytes': FINAL_PLY.stat().st_size,
        'stats_path': str(STATS_PATH),
        'split_stage_reports': FULL_STAGE_REPORTS,
    }
    print(json.dumps(MODEL_AUDIT, indent=2, sort_keys=True))
    disk = shutil.disk_usage(WORK_ROOT)
    print('Working disk free before render GiB:', round(disk.free / 2**30, 2))
else:
    print('Model audit skipped: no 30K model yet (valid gate-only state).')

In [ ]:
VARIANT_SIZES = {}
RENDER_READY = False
if FULL_MODEL_READY:
    if VARIANT_ROOT.exists():
        shutil.rmtree(VARIANT_ROOT)
    VARIANT_ROOT.mkdir(parents=True)
    render_command = [
        sys.executable, 'render_scene.py',
        '--model_dir', str(MODEL_ROOT),
        '--input_dir', str(CLEAN_ROOT),
        '--image_dir', str(VARIANT_ROOT),
        '--orig_dir', str(PUBLIC_ROOT),
        '--scene_name', SCENE,
        '--iterations', str(FINAL_ITERATION),
        '--render_variants',
        '--variant_sharpen_amount', str(SHARPEN_AMOUNT),
        '--sharpen_sigma', str(SHARPEN_SIGMA),
        '--jpeg_quality', str(JPEG_QUALITY),
        '--jpeg_subsampling', str(JPEG_SUBSAMPLING),
        '--jpeg_optimize',
    ]
    print('Running:', ' '.join(render_command))
    subprocess.run(render_command, cwd=REPO, check=True)
    for variant in VARIANT_NAMES:
        scene_dir = VARIANT_ROOT / variant / SCENE
        files = sorted(path for path in scene_dir.iterdir() if path.is_file())
        assert len(files) == 60, '{}: expected 60 files, got {}'.format(variant, len(files))
        total_bytes = sum(path.stat().st_size for path in files)
        VARIANT_SIZES[variant] = total_bytes
        print(
            '{}: {:.3f} MB / 60; estimated 434 = {:.3f} MB'.format(
                variant, total_bytes / 1_000_000, total_bytes / 60 * 434 / 1_000_000
            )
        )
    RENDER_READY = True
else:
    print('Rendering skipped: gate-only state has no 30K model.')

In [ ]:
RESULTS = []
FEASIBLE_RESULTS = []
BEST_SCORE_RESULT = None
BEST_FEASIBLE_RESULT = None
SELECTED_RESULT = None
PRIMARY_RESULT = None
RAW_RESULT = None
PRIMARY_DELTA = None
RANKING_PATH = EVAL_ROOT / '{}_mcmc_q96_results.json'.format(SCENE)
if RENDER_READY:
    if EVAL_ROOT.exists():
        shutil.rmtree(EVAL_ROOT)
    EVAL_ROOT.mkdir(parents=True)
    for variant in VARIANT_NAMES:
        variant_eval_root = EVAL_ROOT / variant
        eval_command = [
            sys.executable, 'eval_scene.py',
            '--input_dir', str(CLEAN_ROOT),
            '--image_dir', str(VARIANT_ROOT / variant),
            '--eval_dir', str(variant_eval_root),
            '--scene_name', SCENE,
            '--psnr_max', '40',
        ]
        subprocess.run(eval_command, cwd=REPO, check=True)
        result_path = variant_eval_root / '{}.json'.format(SCENE)
        result = json.loads(result_path.read_text(encoding='utf-8'))
        assert result['num_images'] == 60
        for metric in ('SSIM', 'PSNR', 'LPIPS', 'weighted_score'):
            assert math.isfinite(float(result[metric]))
        result['variant'] = variant
        result['size_mb_60'] = VARIANT_SIZES[variant] / 1_000_000
        result['estimated_mb_434'] = VARIANT_SIZES[variant] / 60 * 434 / 1_000_000
        result['within_safety_cap_340mb'] = result['estimated_mb_434'] <= SAFETY_CAP_MB
        result['within_hard_limit_350mb'] = result['estimated_mb_434'] <= HARD_LIMIT_MB
        if not result['within_hard_limit_350mb']:
            warnings.warn(
                '{} estimates {:.2f} MB, above the hard 350 MB warning line.'.format(
                    variant, result['estimated_mb_434']
                ), stacklevel=1,
            )
        elif not result['within_safety_cap_340mb']:
            warnings.warn(
                '{} estimates {:.2f} MB: under 350 MB but outside the 340 MB safety cap.'.format(
                    variant, result['estimated_mb_434']
                ), stacklevel=1,
            )
        RESULTS.append(result)

    RESULTS.sort(key=lambda item: item['weighted_score'], reverse=True)
    FEASIBLE_RESULTS = [item for item in RESULTS if item['within_safety_cap_340mb']]
    BEST_SCORE_RESULT = RESULTS[0]
    BEST_FEASIBLE_RESULT = FEASIBLE_RESULTS[0] if FEASIBLE_RESULTS else None
    SELECTED_RESULT = BEST_FEASIBLE_RESULT
    for rank, result in enumerate(RESULTS, 1):
        print(
            '{}. {}: score={:.6f}, PSNR={:.6f}, SSIM={:.6f}, LPIPS={:.6f}, '
            'est434={:.2f}MB, safe340={}'.format(
                rank, result['variant'], result['weighted_score'], result['PSNR'],
                result['SSIM'], result['LPIPS'], result['estimated_mb_434'],
                result['within_safety_cap_340mb'],
            )
        )
    print('Best score without size filter:', BEST_SCORE_RESULT['variant'])
    if SELECTED_RESULT is None:
        warnings.warn(
            'No variant is feasible under the 340 MB safety cap; no submission ZIP will be selected.',
            stacklevel=1,
        )
    else:
        assert SELECTED_RESULT['estimated_mb_434'] <= SAFETY_CAP_MB
        print(
            'Selected best feasible variant: {} (score {:.6f}, estimated {:.2f} MB / 434)'.format(
                SELECTED_RESULT['variant'], SELECTED_RESULT['weighted_score'],
                SELECTED_RESULT['estimated_mb_434'],
            )
        )
    PRIMARY_RESULT = next(item for item in RESULTS if item['variant'] == 'bicubic_sharp0p3')
    RAW_RESULT = next(item for item in RESULTS if item['variant'] == 'bicubic_sharp0')
    PRIMARY_DELTA = {
        key: PRIMARY_RESULT[key] - BASELINE_Q96[key]
        for key in ('SSIM', 'PSNR', 'LPIPS', 'weighted_score', 'estimated_mb_434')
    }
    print('Primary delta versus Improved-GS Q96:')
    print(json.dumps(PRIMARY_DELTA, indent=2, sort_keys=True))
    atomic_write_json(RANKING_PATH, RESULTS)
else:
    print('Evaluation skipped: no rendered 30K variants.')

In [ ]:
EXPORT_ROOT.mkdir(parents=True, exist_ok=True)
FINAL_MANIFEST = {
    **EXPERIMENT_MANIFEST,
    'updated_utc': datetime.now(timezone.utc).isoformat(),
    'run_status': {
        'smoke_completed': SMOKE_REPORT is not None,
        'resource_gate_9000_passed': RESOURCE_GATE_PASSED,
        'continue_after_resource_gate': CONTINUE_AFTER_RESOURCE_GATE,
        'full_model_ready': FULL_MODEL_READY,
        'render_ready': RENDER_READY,
    },
    'smoke_report': SMOKE_REPORT,
    'gate_report': GATE_REPORT,
    'model_audit': MODEL_AUDIT,
    'render': {
        'jpeg_quality': JPEG_QUALITY,
        'jpeg_subsampling': JPEG_SUBSAMPLING,
        'jpeg_optimize': True,
        'sharpen_sigma': SHARPEN_SIGMA,
        'variant_sharpen_amount': SHARPEN_AMOUNT,
        'safety_cap_mb': SAFETY_CAP_MB,
        'hard_warning_mb': HARD_LIMIT_MB,
    },
    'best_score_result': BEST_SCORE_RESULT,
    'best_feasible_result': BEST_FEASIBLE_RESULT,
    'selected_result': SELECTED_RESULT,
    'primary_result': PRIMARY_RESULT,
    'raw_bicubic_result': RAW_RESULT,
    'primary_delta_vs_improvedgs_q96': PRIMARY_DELTA,
}
FINAL_MANIFEST_PATH = EXPORT_ROOT / 'experiment_manifest.json'
atomic_write_json(FINAL_MANIFEST_PATH, FINAL_MANIFEST)
if RANKING_PATH.is_file():
    shutil.copy2(RANKING_PATH, EXPORT_ROOT / RANKING_PATH.name)
if STATS_PATH.is_file():
    shutil.copy2(STATS_PATH, EXPORT_ROOT / STATS_PATH.name)
if RESUME_MANIFEST_PATH.is_file():
    shutil.copy2(RESUME_MANIFEST_PATH, EXPORT_ROOT / 'resume_manifest.json')

if SELECTED_RESULT is not None:
    BEST_VARIANT = SELECTED_RESULT['variant']
    best_scene_dir = VARIANT_ROOT / BEST_VARIANT / SCENE
    best_zip_base = EXPORT_ROOT / 'mcmc_{}_{}_q{}_444'.format(
        SCENE, BEST_VARIANT, JPEG_QUALITY
    )
    best_zip = Path(shutil.make_archive(str(best_zip_base), 'zip', root_dir=best_scene_dir))
    print('Best feasible render ZIP:', best_zip, '({:.2f} MB)'.format(best_zip.stat().st_size / 1_000_000))
elif FULL_MODEL_READY:
    print('No render ZIP exported because no result satisfies the 340 MB safety cap.')

if EXPORT_FINAL_MODEL:
    assert FULL_MODEL_READY, 'A final model can only be exported after 30K.'
    model_archive = EXPORT_ROOT / 'mcmc_{}_model_iter{}.tar.gz'.format(SCENE, FINAL_ITERATION)
    with tarfile.open(model_archive, 'w:gz') as archive:
        archive.add(MODEL_SCENE / 'cfg_args', arcname='{}/cfg_args'.format(SCENE))
        archive.add(MODEL_SCENE / 'mcmc_config.json', arcname='{}/mcmc_config.json'.format(SCENE))
        archive.add(
            FINAL_PLY,
            arcname='{}/point_cloud/iteration_{}/point_cloud.ply'.format(SCENE, FINAL_ITERATION),
        )
    print('Final model archive:', model_archive)

if EXPORT_RESUME_BUNDLE:
    checkpoint_iter, checkpoint_path = latest_checkpoint(MODEL_SCENE)
    assert checkpoint_path is not None and RESUME_MANIFEST_PATH.is_file()
    resume_manifest = validate_local_resume(checkpoint_path)
    resume_archive = EXPORT_ROOT / 'mcmc_{}_resume_{}.zip'.format(SCENE, checkpoint_iter)
    with zipfile.ZipFile(
        resume_archive, 'w', compression=zipfile.ZIP_STORED, allowZip64=True
    ) as archive:
        archive.write(checkpoint_path, checkpoint_path.name)
        archive.write(RESUME_MANIFEST_PATH, 'resume_manifest.json')
        archive.write(EXPERIMENT_MANIFEST_PATH, 'experiment_manifest.json')
        if STATS_PATH.is_file() and resume_manifest.get('stats_file'):
            archive.write(STATS_PATH, resume_manifest['stats_file'])
    print('Resume ZIP (directory/ZIP importer compatible):', resume_archive)

if RESOURCE_GATE_PASSED and not FULL_MODEL_READY:
    print('Gate-only artifacts are valid. Checkpoint remains at:', latest_checkpoint(MODEL_SCENE)[1])
    print('Set EXPORT_RESUME_BUNDLE=True only when a portable checkpoint ZIP is needed.')
print('Export files:')
for path in sorted(EXPORT_ROOT.iterdir()):
    if path.is_file():
        print(' - {}: {:.3f} MB'.format(path.name, path.stat().st_size / 1_000_000))
print('Save a Kaggle notebook version with outputs before ending the session.')